In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ----------------------------
# Paths
# ----------------------------
images_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/imagesTs")  
labels_folder = Path("/data/colon_cancer/Classifier/Decathlon/raw_splitted/labelsTs")  
output_csv = Path("/data/colon_cancer/Classifier/Decathlon/labels.csv") 
"""
for img_path in images_folder.glob("*0000.nii.gz"): 
    # Extract number from filename
    number = img_path.name.replace("_0000.nii.gz", "")
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"._colon_{number_int}.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
"""


# ----------------------------
# Rename images
# ----------------------------
renamed_images = []

for img_path in images_folder.glob("colon_*.nii.gz"):
    # Extract number from filename
    number = img_path.name.replace("colon_", "").replace(".nii.gz", "")  
    number_int = int(number)  # convert to int to remove leading zeros
    
    new_name = f"{number_int}_0000.nii.gz"  # e.g., 12_0000.nii
    new_path = images_folder / new_name
    shutil.move(str(img_path), str(new_path))
    
    renamed_images.append(new_path)

# ----------------------------
# Rename labels
# ----------------------------
for lbl_path in labels_folder.glob("colon_*.nii.gz"):
    number = lbl_path.name.replace("colon_", "").replace(".nii.gz", "")  # '001' from 'colon_001'
    number_int = int(number)  # convert to int
    new_name = f"{number_int}.nii.gz"
    new_path = labels_folder / new_name
    shutil.move(str(lbl_path), str(new_path))

# ----------------------------
# Generate CSV
# ----------------------------
data = []

for img_path in sorted(images_folder.glob("*_0000.nii.gz")):
    uid = img_path.stem.split("_")[0]  # get the UID (number before '_0000')
    data.append({
        "UID": uid,
        "img_path": str(img_path),
        "target": 1,
        "Split": "test",
        "Fold": 0
    })

df = pd.DataFrame(data, columns=["UID", "img_path", "target", "Split", "Fold"])
df.to_csv(output_csv, index=False)
print(f"CSV saved to {output_csv}")


In [2]:
#####################  discard T1 samples #####################################
import os
import shutil

# =========================
# CONFIG
# =========================

src_images_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr"
src_labels_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTr"

dst_images_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded"
dst_labels_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded"

uids = [
    "525",
    "547",
    "656",
    "551",
    "163",
    "569",
    "529",
    "556",
    "563",
    "527",
    "566",
    "586",
    "571",
    "580",
    "562",
    "525",
    "548",
    "275",
    "510",
    "547",
    "521",
    "575",
    "531",
    "589",
    "577",
    "514",
    "541",
    "553",
    "539",
    "561",
    "505",
    "540",
    "532",
    "517",
    "522",
    "512",
    "593",
    "504",
    "516",
    "587",
]

# =========================
# SETUP
# =========================

os.makedirs(dst_images_dir, exist_ok=True)
os.makedirs(dst_labels_dir, exist_ok=True)

# =========================
# MOVE FILES
# =========================

for uid in uids:
    img_name = f"{uid}_0000.nii.gz"
    lbl_name = f"{uid}.nii.gz"

    src_img = os.path.join(src_images_dir, img_name)
    src_lbl = os.path.join(src_labels_dir, lbl_name)

    dst_img = os.path.join(dst_images_dir, img_name)
    dst_lbl = os.path.join(dst_labels_dir, lbl_name)

    print(f"\nProcessing UID: {uid}")

    # --- Image ---
    if os.path.exists(src_img):
        shutil.move(src_img, dst_img)
        print(f"  ✓ Moved image → {dst_img}")
    else:
        print(f"  ⚠ Image not found: {src_img}")

    # --- Label ---
    if os.path.exists(src_lbl):
        shutil.move(src_lbl, dst_lbl)
        print(f"  ✓ Moved label → {dst_lbl}")
    else:
        print(f"  ⚠ Label not found: {src_lbl}")

print("\nDone.")



Processing UID: 525
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/525_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/525.nii.gz

Processing UID: 547
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/547_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/547.nii.gz

Processing UID: 656
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/656_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/656.nii.gz

Processing UID: 551
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/images_excluded/551_0000.nii.gz
  ✓ Moved label → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels_excluded/551.nii.gz

Processing UID: 163
  ✓ Moved image → /data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/im

In [ ]:
################################## clean labels csv #####################################
import pandas as pd

csv_path = "/data/colon_cancer/Classifier/ColonCancer/labels.csv"
output_csv_path = "/data/colon_cancer/Classifier/ColonCancer/labels_cleaned.csv"

# UIDs to remove (convert once, remove duplicates)
uids = [
    525, 547, 656, 551, 163, 569, 529, 556, 563, 527, 566,
    586, 571, 580, 562, 548, 275, 510, 521, 575, 531, 589,
    577, 514, 541, 553, 539, 561, 505, 540, 532, 517, 522,
    512, 593, 504, 516, 587
]
uids = set(uids)  # deduplicate + faster lookup

# Load CSV
df = pd.read_csv(csv_path)

# Normalize UID column
df["UID"] = df["UID"].astype(str).str.strip().astype(int)

before = len(df)

# Debug: check overlap
overlap = set(df["UID"]).intersection(uids)
print(f"UIDs found in CSV to remove: {sorted(overlap)}")

# Remove rows
df_clean = df[~df["UID"].isin(uids)]

after = len(df)

# Save cleaned CSV
df_clean.to_csv(output_csv_path, index=False)

print(f"Removed {before - after} rows")
print(f"Saved cleaned CSV to: {output_csv_path}")


In [ ]:
########################## split dataset ###############################
import pandas as pd
import numpy as np

# -------------------------
# Config
# -------------------------
INPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/labels_cleaned.csv"
OUTPUT_CSV = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

N_TRAIN = 600
N_VAL   = 118
N_TEST  = 77

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -------------------------
# Load data
# -------------------------
df = pd.read_csv(INPUT_CSV)

assert len(df) == N_TRAIN + N_VAL + N_TEST, "Split sizes do not sum to dataset size!"

# -------------------------
# Compute class proportions
# -------------------------
class_counts = df["target"].value_counts().sort_index()
total = len(df)

class_ratios = class_counts / total

# Samples per class per split
def split_counts(n_total):
    counts = (class_ratios * n_total).round().astype(int)
    # fix rounding errors
    diff = n_total - counts.sum()
    if diff != 0:
        counts.iloc[0] += diff
    return counts

train_counts = split_counts(N_TRAIN)
val_counts   = split_counts(N_VAL)
test_counts  = split_counts(N_TEST)

# -------------------------
# Perform stratified split
# -------------------------
df["split"] = None
remaining_idx = []

for cls in class_counts.index:
    cls_df = df[df["target"] == cls].sample(frac=1, random_state=RANDOM_SEED)

    n_train = train_counts[cls]
    n_val   = val_counts[cls]
    n_test  = test_counts[cls]

    train_idx = cls_df.iloc[:n_train].index
    val_idx   = cls_df.iloc[n_train:n_train + n_val].index
    test_idx  = cls_df.iloc[n_train + n_val:n_train + n_val + n_test].index

    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"]   = "val"
    df.loc[test_idx, "split"]  = "test"

# -------------------------
# Sanity checks
# -------------------------
print("\nSplit sizes:")
print(df["split"].value_counts())

print("\nClass balance per split:")
print(df.groupby(["split", "target"]).size().unstack())

assert df["split"].isna().sum() == 0, "Some samples were not assigned a split!"

# -------------------------
# Save
# -------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved splits to {OUTPUT_CSV}")



Split sizes:
split
train    600
val      118
test      77
Name: count, dtype: int64

Class balance per split:
target    0    1
split           
test     32   45
train   251  349
val      49   69

Saved splits to /data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv


In [ ]:
import os
from pathlib import Path
import SimpleITK as sitk
import pydicom


# ============================
# CONFIG
# ============================

ROOT_DIR = Path("/data/colon_cancer/normal/NomalCT")
OUT_DIR = Path("/data/colon_cancer/normal/nifti")

OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================
# HELPERS
# ============================

def looks_like_dicom(path):
    """Very permissive DICOM check (Sectra-safe)."""
    try:
        pydicom.dcmread(path, stop_before_pixels=True, force=True)
        return True
    except:
        return False


def is_dicom_folder(folder):
    """Detect folders that actually contain DICOM slices."""
    if not folder.is_dir():
        return False

    checked = 0
    for f in folder.iterdir():
        if f.is_file():
            checked += 1
            if looks_like_dicom(f):
                return True
        if checked >= 5:   # do not scan entire folder
            break
    return False


def dicom_folder_to_nifti(dicom_dir, output_path):
    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(dicom_dir)

    if not series_ids:
        raise RuntimeError(f"No DICOM series found in {dicom_dir}")

    series_files = reader.GetGDCMSeriesFileNames(dicom_dir, series_ids[0])
    reader.SetFileNames(series_files)

    image = reader.Execute()
    sitk.WriteImage(image, output_path)


# ============================
# MAIN
# ============================

for patient_dir in sorted(ROOT_DIR.iterdir()):
    if not patient_dir.is_dir():
        continue

    patient_id = patient_dir.name
    dicom_root = patient_dir / "DICOM"

    if not dicom_root.exists():
        print(f"[SKIP] {patient_id}: No DICOM folder")
        continue

    print(f"\nProcessing patient {patient_id}")

    found = False

    for root, dirs, files in os.walk(dicom_root):
        root = Path(root)

        if files and is_dicom_folder(root):
            out_path = OUT_DIR / f"{patient_id}.nii.gz"
            print(f"  → Found DICOM series in {root}")
            dicom_folder_to_nifti(str(root), str(out_path))
            found = True
            break  # one scan per patient

    if not found:
        print(f"  ❌ No DICOM series found for patient {patient_id}")

print("\nDone.")
#8,9,28,33

In [ ]:
from pathlib import Path

nifti_dir = Path("/data/colon_cancer/normal/nifti")

for nifti in nifti_dir.iterdir():
    if nifti.suffix == ".gz" and nifti.name.endswith(".nii.gz"):
        stem = nifti.name[:-7]   # remove .nii.gz
        if stem.endswith("_0000"):
            continue

        new_name = f"{stem}_0000.nii.gz"
        new_path = nifti_dir / new_name

        if new_path.exists():
            print(f"[SKIP] {new_name} already exists")
            continue

        nifti.rename(new_path)
        print(f"Renamed: {nifti.name} → {new_name}")

    elif nifti.suffix == ".nii":
        stem = nifti.stem
        if stem.endswith("_0000"):
            continue

        new_name = f"{stem}_0000.nii"
        new_path = nifti_dir / new_name

        if new_path.exists():
            print(f"[SKIP] {new_name} already exists")
            continue

        nifti.rename(new_path)
        print(f"Renamed: {nifti.name} → {new_name}")


Renamed: 45.nii.gz → 45_0000.nii.gz
Renamed: 48.nii.gz → 48_0000.nii.gz
Renamed: 1.nii.gz → 1_0000.nii.gz
Renamed: 50.nii.gz → 50_0000.nii.gz
Renamed: 23.nii.gz → 23_0000.nii.gz
Renamed: 6.nii.gz → 6_0000.nii.gz
Renamed: 27.nii.gz → 27_0000.nii.gz
Renamed: 14.nii.gz → 14_0000.nii.gz
Renamed: 18.nii.gz → 18_0000.nii.gz
Renamed: 16.nii.gz → 16_0000.nii.gz
Renamed: 40.nii.gz → 40_0000.nii.gz
Renamed: 34.nii.gz → 34_0000.nii.gz
Renamed: 38.nii.gz → 38_0000.nii.gz
Renamed: 21.nii.gz → 21_0000.nii.gz
Renamed: 10.nii.gz → 10_0000.nii.gz
Renamed: 30.nii.gz → 30_0000.nii.gz
Renamed: 46.nii.gz → 46_0000.nii.gz
Renamed: 26.nii.gz → 26_0000.nii.gz
Renamed: 9.nii.gz → 9_0000.nii.gz
Renamed: 2.nii.gz → 2_0000.nii.gz
Renamed: 15.nii.gz → 15_0000.nii.gz
Renamed: 28.nii.gz → 28_0000.nii.gz
Renamed: 29.nii.gz → 29_0000.nii.gz
Renamed: 5.nii.gz → 5_0000.nii.gz
Renamed: 36.nii.gz → 36_0000.nii.gz
Renamed: 3.nii.gz → 3_0000.nii.gz
Renamed: 24.nii.gz → 24_0000.nii.gz
Renamed: 7.nii.gz → 7_0000.nii.gz
Rename

In [7]:
##################### move test samples ###############################import os
import shutil
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Paths (EDIT THESE)
# --------------------------------------------------
splits_csv = "/data/colon_cancer/Classifier/ColonCancer/splits_cleaned.csv"

images_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTr")
labels_src = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTr")

images_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs")
labels_dst = Path("/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTs")

# Create destination folders
images_dst.mkdir(parents=True, exist_ok=True)
labels_dst.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Load CSV
# --------------------------------------------------
df = pd.read_csv(splits_csv)

# Normalize columns
df["UID"] = df["UID"].astype(str)
df["split"] = df["split"].str.lower()

# Select test set
test_uids = df[df["split"] == "test"]["UID"].tolist()

print(f"Found {len(test_uids)} test cases")

# --------------------------------------------------
# Move files
# --------------------------------------------------
moved_images = 0
moved_labels = 0

for uid in test_uids:
    img_name = f"{uid}_0000.nii.gz"
    lbl_name = f"{uid}.nii.gz"

    img_src = images_src / img_name
    lbl_src = labels_src / lbl_name

    img_dst = images_dst / img_name
    lbl_dst = labels_dst / lbl_name

    # Move image
    if img_src.exists():
        shutil.move(str(img_src), str(img_dst))
        moved_images += 1
        print(f"✓ Moved image: {img_name}")
    else:
        print(f"⚠ Image missing: {img_name}")

    # Move label
    if lbl_src.exists():
        shutil.move(str(lbl_src), str(lbl_dst))
        moved_labels += 1
        print(f"✓ Moved label: {lbl_name}")
    else:
        print(f"⚠ Label missing: {lbl_name}")

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("\n=== Summary ===")
print(f"Images moved: {moved_images}")
print(f"Labels moved: {moved_labels}")
print("Done ✅")

Found 77 test cases
✓ Moved image: 334_0000.nii.gz
✓ Moved label: 334.nii.gz
✓ Moved image: 620_0000.nii.gz
✓ Moved label: 620.nii.gz
✓ Moved image: 489_0000.nii.gz
✓ Moved label: 489.nii.gz
✓ Moved image: 761_0000.nii.gz
✓ Moved label: 761.nii.gz
✓ Moved image: 31_0000.nii.gz
✓ Moved label: 31.nii.gz
✓ Moved image: 172_0000.nii.gz
✓ Moved label: 172.nii.gz
✓ Moved image: 357_0000.nii.gz
✓ Moved label: 357.nii.gz
✓ Moved image: 78_0000.nii.gz
✓ Moved label: 78.nii.gz
✓ Moved image: 33_0000.nii.gz
✓ Moved label: 33.nii.gz
✓ Moved image: 795_0000.nii.gz
✓ Moved label: 795.nii.gz
✓ Moved image: 148_0000.nii.gz
✓ Moved label: 148.nii.gz
✓ Moved image: 265_0000.nii.gz
✓ Moved label: 265.nii.gz
✓ Moved image: 628_0000.nii.gz
✓ Moved label: 628.nii.gz
✓ Moved image: 346_0000.nii.gz
✓ Moved label: 346.nii.gz
✓ Moved image: 74_0000.nii.gz
✓ Moved label: 74.nii.gz
✓ Moved image: 728_0000.nii.gz
✓ Moved label: 728.nii.gz
✓ Moved image: 176_0000.nii.gz
✓ Moved label: 176.nii.gz
✓ Moved image: 499_

In [8]:
import pandas as pd

# Load Excel file
df = pd.read_csv("train_reports.csv")

# Basic checks
print("Columns:", df.columns.tolist())
print("Number of reports:", len(df))

# Peek at one row
print("\n--- Example row ---")
print(df.iloc[0])

print("Number of reports:", len(df))
print(df.columns)


Columns: ['VolumeName', 'ClinicalInformation_EN', 'Technique_EN', 'Findings_EN', 'Impressions_EN']
Number of reports: 47149

--- Example row ---
VolumeName                                               train_1_a_1.nii.gz
ClinicalInformation_EN                                           Not given.
Technique_EN              Non-contrast images were taken in the axial pl...
Findings_EN               Multiple venous collaterals are present in the...
Impressions_EN             Multiple venous collaterals in the anterior l...
Name: 0, dtype: object
Number of reports: 47149
Index(['VolumeName', 'ClinicalInformation_EN', 'Technique_EN', 'Findings_EN',
       'Impressions_EN'],
      dtype='object')


In [16]:
import pandas as pd

# =========================
# 1. Load CSV
# =========================
CSV_PATH = "train_reports.csv"

df = pd.read_csv(CSV_PATH)

print("Loaded CT-RATE reports")
print("Number of reports:", len(df))
print("Columns:", df.columns.tolist())

# =========================
# 2. Define columns
# =========================
ID_COL = "VolumeName"
IMPR_COL = "Impressions_EN"
FIND_COL = "Findings_EN"

df[IMPR_COL] = df[IMPR_COL].fillna("")
df[FIND_COL] = df[FIND_COL].fillna("")

df["impr_clean"] = df[IMPR_COL].str.lower()
df["find_clean"] = df[FIND_COL].str.lower()

# =========================
# 3. Define string-matching terms
# =========================
colon_cancer_terms = [
    "colon cancer",
    "colonic cancer",
    "colorectal cancer",
    "colon carcinoma",
    "colonic carcinoma",
    "colorectal carcinoma",
    "colon ca.",
]

diverticulitis_terms = [
    "diverticulitis",
    "diverticular disease",
    "sigmoid diverticulitis",
    "acute diverticulitis",
]

# =========================
# 4. Helper functions
# =========================
def contains_any(text, terms):
    return any(term in text for term in terms)

# =========================
# 5. Initial string matching
# =========================
df["colon_any"] = (
    df["impr_clean"].apply(lambda x: contains_any(x, colon_cancer_terms)) |
    df["find_clean"].apply(lambda x: contains_any(x, colon_cancer_terms))
)

df["divert_any"] = (
    df["impr_clean"].apply(lambda x: contains_any(x, diverticulitis_terms)) |
    df["find_clean"].apply(lambda x: contains_any(x, diverticulitis_terms))
)

print("\nInitial matches:")
print("Colon cancer:", df["colon_any"].sum())
print("Diverticulitis:", df["divert_any"].sum())

# =========================
# 6. Remove negated diverticulitis cases
# =========================
negation_phrases = [
    "no diverticulitis",
    "no signs of diverticulitis",
    "no sign of diverticulitis",
    "no evidence of diverticulitis",
    "without diverticulitis",
    "diverticulitis excluded",
    "diverticulitis is excluded",
    "no findings in favor of diverticulitis",
    "no finding in favor of diverticulitis",
]

def contains_negation(text, neg_terms):
    return any(neg in text for neg in neg_terms)

df["divert_negated"] = (
    df["impr_clean"].apply(lambda x: contains_negation(x, negation_phrases)) |
    df["find_clean"].apply(lambda x: contains_negation(x, negation_phrases))
)

df["divert_positive"] = df["divert_any"] & (~df["divert_negated"])

print("\nDiverticulitis cleaning:")
print("Negated cases removed:", df["divert_negated"].sum())
print("Final diverticulitis cases:", df["divert_positive"].sum())

# =========================
# 7. Select columns to save
# =========================
cols_to_save = [ID_COL, IMPR_COL, FIND_COL]

colon_df = df[df["colon_any"]][cols_to_save]
divert_df = df[df["divert_positive"]][cols_to_save]

# =========================
# 8. Save cleaned outputs
# =========================
colon_df.to_csv(
    "ct_rate_colon_cancer_reports_clean.csv", index=False
)

divert_df.to_csv(
    "ct_rate_diverticulitis_reports_clean.csv", index=False
)

print("\nSaved output files:")
print("- ct_rate_colon_cancer_reports_clean.csv")
print("- ct_rate_diverticulitis_reports_clean.csv")


Loaded CT-RATE reports
Number of reports: 47149
Columns: ['VolumeName', 'ClinicalInformation_EN', 'Technique_EN', 'Findings_EN', 'Impressions_EN']

Initial matches:
Colon cancer: 39
Diverticulitis: 56

Diverticulitis cleaning:
Negated cases removed: 44
Final diverticulitis cases: 12

Saved output files:
- ct_rate_colon_cancer_reports_clean.csv
- ct_rate_diverticulitis_reports_clean.csv


In [6]:
import pandas as pd

# =========================
# 1. Load CSV
# =========================
CSV_PATH = "train_reports.csv"

df = pd.read_csv(CSV_PATH)

print("Loaded CT-RATE reports")
print("Number of reports:", len(df))
print("Columns:", df.columns.tolist())

# =========================
# 2. Define text columns
# =========================
IMPR_COL = "Impressions_EN"
FIND_COL = "Findings_EN"

df[IMPR_COL] = df[IMPR_COL].fillna("")
df[FIND_COL] = df[FIND_COL].fillna("")

df["impr_clean"] = df[IMPR_COL].str.lower()
df["find_clean"] = df[FIND_COL].str.lower()

print("Empty impressions:", (df["impr_clean"] == "").sum())
print("Empty findings:", (df["find_clean"] == "").sum())

print("\n--- Example impression ---")
print(df.iloc[0][IMPR_COL])

print("\n--- Example finding ---")
print(df.iloc[0][FIND_COL])

print("\n--- Example ClinicalInformation  ---")
print(df.iloc[500]["ClinicalInformation_EN"])

print("\n--- Example Technique  ---")
print(df.iloc[0]["Technique_EN"])



Loaded CT-RATE reports
Number of reports: 47149
Columns: ['VolumeName', 'ClinicalInformation_EN', 'Technique_EN', 'Findings_EN', 'Impressions_EN']
Empty impressions: 28
Empty findings: 2

--- Example impression ---
 Multiple venous collaterals in the anterior left chest wall and collapsed appearance in the left subclavian vein (chronic occlusion?).  Thickening of the bronchial wall in both lungs.  Peribronchial reticulonodular densities in the lower lobes, minimal consolidations (infection process?).  Atelectasis in both lungs.  Thoracic spondylosis.

--- Example finding ---
Multiple venous collaterals are present in the anterior left chest wall and are associated with the anterior jugular vein at the level of the right sternoclavicular junction. Left subclavian vein collapsed (chronic occlusion pathology?). Trachea, both main bronchi are open. Calcific plaques are observed in the aortic arch. Other mediastinal main vascular structures, heart contour, size are normal. Thoracic aorta di

In [ ]:
import pandas as pd

# Load metadata CSV
df = pd.read_csv("metadata_3.csv")

# Keep only abdomen CT scans
abdomen_df = df[
    df["Study Description"].str.contains("Abdomen", case=False, na=False)
]

# Optional: also ensure modality is CT
abdomen_df = abdomen_df[abdomen_df["Modality"] == "CT"]

print(abdomen_df[["Subject ID", "Series Description"]])
print(f"Total abdomen CT scans: {len(abdomen_df)}")




                   Subject ID       Series Description
0   StageII-Colorectal-CT-102  Venous Phase  5.0  B30f
3   StageII-Colorectal-CT-101                      NaN
7   StageII-Colorectal-CT-108                      NaN
10  StageII-Colorectal-CT-112  Venous Phase  5.0  B30f
11  StageII-Colorectal-CT-113  Venous Phase  5.0  B30f
12  StageII-Colorectal-CT-114  Venous Phase  5.0  B30f
22  StageII-Colorectal-CT-122                      NaN
23  StageII-Colorectal-CT-123  Venous Phase  5.0  B30f
26  StageII-Colorectal-CT-126                      NaN
32  StageII-Colorectal-CT-134  Venous Phase  5.0  B30f
36  StageII-Colorectal-CT-137                      NaN
37  StageII-Colorectal-CT-138  Venous Phase  5.0  B30f
40  StageII-Colorectal-CT-142  Venous Phase  5.0  B30f
42  StageII-Colorectal-CT-143  Venous Phase  5.0  B30f
44  StageII-Colorectal-CT-146                      NaN
45  StageII-Colorectal-CT-144  Venous Phase  5.0  B30f
47  StageII-Colorectal-CT-148  Venous Phase  5.0  B30f
Total abdo

In [1]:
import os
import dicom2nifti
from pathlib import Path

# =========================
# CONFIG
# =========================

DICOM_ROOT = Path("/data/colon_cancer/Task101_Colon/StageII-Colorectal-CT")
OUTPUT_ROOT = Path("/data/colon_cancer/Task101_Colon/StageII-Colorectal-CT-nifti")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional: avoid crashing on minor DICOM issues
dicom2nifti.settings.disable_validate_slice_increment()
dicom2nifti.settings.disable_validate_orthogonal()
dicom2nifti.settings.disable_validate_slicecount()

# =========================
# HELPERS
# =========================

def contains_dicom_files(folder: Path) -> bool:
    return any(f.suffix.lower() == ".dcm" for f in folder.iterdir() if f.is_file())

# =========================
# MAIN LOOP
# =========================

for root, dirs, files in os.walk(DICOM_ROOT):
    root = Path(root)

    if not any(f.lower().endswith(".dcm") for f in files):
        continue

    # Identify patient ID from top-level folder
    try:
        patient_id = root.relative_to(DICOM_ROOT).parts[0]
    except Exception:
        patient_id = root.name

    out_file = OUTPUT_ROOT / f"{patient_id}.nii.gz"

    if out_file.exists():
        print(f"[SKIP] {out_file.name} already exists")
        continue

    try:
        print(f"[INFO] Converting {root}")
        dicom2nifti.convert_directory(
            dicom_directory=str(root),
            output_folder=str(OUTPUT_ROOT),
            compression=True,
            reorient=True
        )

        # Rename output (dicom2nifti uses random names)
        generated = sorted(OUTPUT_ROOT.glob("*.nii.gz"), key=os.path.getmtime)
        if generated:
            generated[-1].rename(out_file)
            print(f"[OK] Saved {out_file.name}")

    except Exception as e:
        print(f"[ERROR] Failed for {root}")
        print(e)

print("Conversion finished.")


[INFO] Converting /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT/StageII-Colorectal-CT-146/01-27-2007-NA-Abdomen CT RoutineEnhanced-11383/5.000000-NA-22113
[OK] Saved StageII-Colorectal-CT-146.nii.gz
[INFO] Converting /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT/StageII-Colorectal-CT-108/06-21-2006-NA-Abdomen CT RoutineEnhanced-43586/4.000000-NA-12325
[OK] Saved StageII-Colorectal-CT-108.nii.gz
[INFO] Converting /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT/StageII-Colorectal-CT-048/07-12-2005-NA-Pelvic Cavity CT RoutineEnhanced-44159/2.000000-Pelvis  5.0  B31f-03439
[OK] Saved StageII-Colorectal-CT-048.nii.gz
[INFO] Converting /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT/StageII-Colorectal-CT-083/01-24-2006-NA-penqctzengq-59865/2.000000-Pelvis  5.0  B31f-20208
[OK] Saved StageII-Colorectal-CT-083.nii.gz
[INFO] Converting /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT/StageII-Colorectal-CT-131/11-09-2006-NA-Pelvic Cavity CT RoutineEnhanced-8

In [2]:
import re
from pathlib import Path

# =========================
# CONFIG
# =========================

NIFTI_DIR = Path("/data/colon_cancer/Task101_Colon/StageII-Colorectal-CT-nifti")

# Regex to extract UID (last number in filename)
UID_REGEX = re.compile(r"(\d+)(?=\.nii\.gz$)")

# =========================
# MAIN
# =========================

for f in sorted(NIFTI_DIR.glob("*.nii.gz")):
    match = UID_REGEX.search(f.name)

    if not match:
        print(f"[SKIP] Could not extract UID from {f.name}")
        continue

    uid = match.group(1)
    new_name = f"{uid}_0000.nii.gz"
    new_path = NIFTI_DIR / new_name

    if new_path.exists():
        raise RuntimeError(f"Target file already exists: {new_path}")

    print(f"{f.name}  ->  {new_name}")
    f.rename(new_path)

print("Renaming complete.")


StageII-Colorectal-CT-001.nii.gz  ->  001_0000.nii.gz
StageII-Colorectal-CT-002.nii.gz  ->  002_0000.nii.gz
StageII-Colorectal-CT-003.nii.gz  ->  003_0000.nii.gz
StageII-Colorectal-CT-004.nii.gz  ->  004_0000.nii.gz
StageII-Colorectal-CT-005.nii.gz  ->  005_0000.nii.gz
StageII-Colorectal-CT-006.nii.gz  ->  006_0000.nii.gz
StageII-Colorectal-CT-007.nii.gz  ->  007_0000.nii.gz
StageII-Colorectal-CT-008.nii.gz  ->  008_0000.nii.gz
StageII-Colorectal-CT-009.nii.gz  ->  009_0000.nii.gz
StageII-Colorectal-CT-010.nii.gz  ->  010_0000.nii.gz
StageII-Colorectal-CT-011.nii.gz  ->  011_0000.nii.gz
StageII-Colorectal-CT-012.nii.gz  ->  012_0000.nii.gz
StageII-Colorectal-CT-013.nii.gz  ->  013_0000.nii.gz
StageII-Colorectal-CT-014.nii.gz  ->  014_0000.nii.gz
StageII-Colorectal-CT-015.nii.gz  ->  015_0000.nii.gz
StageII-Colorectal-CT-016.nii.gz  ->  016_0000.nii.gz
StageII-Colorectal-CT-017.nii.gz  ->  017_0000.nii.gz
StageII-Colorectal-CT-018.nii.gz  ->  018_0000.nii.gz
StageII-Colorectal-CT-019.ni

In [3]:
import re
from pathlib import Path

# =========================
# CONFIG
# =========================

NIFTI_DIR = Path("/data/colon_cancer/Task101_Colon/StageII-Colorectal-CT-nifti")

# Match leading zeros in the UID part
PATTERN = re.compile(r"^0*(\d+)(_0000\.nii\.gz)$")

# =========================
# MAIN
# =========================

for f in sorted(NIFTI_DIR.glob("*_0000.nii.gz")):
    match = PATTERN.match(f.name)

    if not match:
        print(f"[SKIP] {f.name}")
        continue

    uid = match.group(1)   # UID without leading zeros
    suffix = match.group(2)

    new_name = f"{uid}{suffix}"
    new_path = NIFTI_DIR / new_name

    if new_path.exists():
        raise RuntimeError(f"Target file already exists: {new_path}")

    print(f"{f.name}  ->  {new_name}")
    f.rename(new_path)

print("Done. Leading zeros removed.")


001_0000.nii.gz  ->  1_0000.nii.gz
002_0000.nii.gz  ->  2_0000.nii.gz
003_0000.nii.gz  ->  3_0000.nii.gz
004_0000.nii.gz  ->  4_0000.nii.gz
005_0000.nii.gz  ->  5_0000.nii.gz
006_0000.nii.gz  ->  6_0000.nii.gz
007_0000.nii.gz  ->  7_0000.nii.gz
008_0000.nii.gz  ->  8_0000.nii.gz
009_0000.nii.gz  ->  9_0000.nii.gz
010_0000.nii.gz  ->  10_0000.nii.gz
011_0000.nii.gz  ->  11_0000.nii.gz
012_0000.nii.gz  ->  12_0000.nii.gz
013_0000.nii.gz  ->  13_0000.nii.gz
014_0000.nii.gz  ->  14_0000.nii.gz
015_0000.nii.gz  ->  15_0000.nii.gz
016_0000.nii.gz  ->  16_0000.nii.gz
017_0000.nii.gz  ->  17_0000.nii.gz
018_0000.nii.gz  ->  18_0000.nii.gz
019_0000.nii.gz  ->  19_0000.nii.gz
020_0000.nii.gz  ->  20_0000.nii.gz
021_0000.nii.gz  ->  21_0000.nii.gz
022_0000.nii.gz  ->  22_0000.nii.gz
023_0000.nii.gz  ->  23_0000.nii.gz
024_0000.nii.gz  ->  24_0000.nii.gz
025_0000.nii.gz  ->  25_0000.nii.gz
026_0000.nii.gz  ->  26_0000.nii.gz
027_0000.nii.gz  ->  27_0000.nii.gz
028_0000.nii.gz  ->  28_0000.nii.gz
0

RuntimeError: Target file already exists: /data/colon_cancer/Task101_Colon/StageII-Colorectal-CT-nifti/100_0000.nii.gz